# DX702 Coding Quiz: Week 6

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors

# import scipy.stats as stats
# import seaborn as sns

# from scipy.stats import skew
# from sklearn.linear_model import LinearRegression

## Task

In this dataset, match treated (X = 1) to untreated (X = 0) based on the confounder (Z). 

Find the **average treatment effect** (each item corresponds to one counterfactual) where the counterfactual is the nearest item in the other group (you can use NearestNeighbors for this.) 

Then, find the **average treatment effect** on the *treated*, where each treated item corresponds to a counterfactual untreated item, but we otherwise ignore the untreated items. 

Then, find the **average treatment effect** on the *untreated*, where each untreated item corresponds to a counterfactual treated item, but we otherwise ignore the treated items. 

Finally, find the **marginal treatment effect**, which is the maximum treatment effect across all untreated items (i.e., it ends up considering only a single untreated item with its single counterfactual). 

In [3]:

df_6 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/main/homework_6.1.csv')
df_6.drop(columns=['Unnamed: 0'], inplace = True)
print(df_6.info())
df_6.head(n = 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Z       1000 non-null   float64
 1   X       1000 non-null   int64  
 2   Y       1000 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 23.6 KB
None


,Z,X,Y
0,0.548814,0,-0.823220
1,0.715189,1,0.842405
2,0.602763,1,0.898618
3,0.544883,0,-0.817325
4,0.423655,0,-0.635482
5,0.645894,0,-0.968841
6,0.437587,0,-0.656381
7,0.891773,1,0.754113
8,0.963663,1,0.718169
9,0.383442,1,1.008279



<font color='plum'>Using k-Nearest Neighbors algorithm to find the closest counterfactual for each data point based on the confounder, Z. The process:

#### 1. Separate the data by splitting into 'treated' group (X=1) and 'untreated' group (X=0).
#### 2. Find Counterfactuals:
- To find the Average Treatment Effect on the Treated (ATT),  identify the nearest untreated neighbor for each treated subject based on the 'Z' value.
- To find the Average Treatment Effect on the Untreated (ATU),  find the nearest treated neighbor for each untreated subject.
#### 3. Calculate Treatment Effects:
 - Individual Treatment Effect (ITE) = the difference in the outcome 'Y' between a subject and its counterfactual (Y_treated - Y_untreated).
 - ATT is the average of the ITEs for all treated subjects.
 - ATU is the average of the ITEs for all untreated subjects.
 - Average Treatment Effect (ATE) is the overall average of all calculated ITEs from both the treated and untreated groups.
 - Marginal Treatment Effect (MTE), as defined in the quiz, is the maximum ITE found among the untreated group.

In [ ]:
# Separate into treated and untreated groups
treated_df      = df_6[df_6['X'] == 1].reset_index(drop=True)
untreated_df    = df_6[df_6['X'] == 0].reset_index(drop=True)

# Reshape confounder data for NearestNeighbors
Z_treated   = treated_df[['Z']]
Z_untreated = untreated_df[['Z']]

# --- ATT Calculation ---
# Find the nearest untreated neighbor for each treated subject
nn_untreated = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(Z_untreated)
distances_att, indices_att = nn_untreated.kneighbors(Z_treated)

# Get outcomes of the matched untreated subjects
y_counterfactual_att = untreated_df.loc[indices_att.flatten(), 'Y'].values

#  individual effects for the treated
effects_for_treated = treated_df['Y'] - y_counterfactual_att

#  ATT
att = effects_for_treated.mean()

# --- ATU Calculation ---
# Find  nearest treated neighbor for each untreated subject
nn_treated = NearestNeighbors(n_neighbors = 1, algorithm = 'auto').fit(Z_treated)
distances_atu, indices_atu = nn_treated.kneighbors(Z_untreated)

# Get  outcomes of the matched treated subjects
y_counterfactual_atu = treated_df.loc[indices_atu.flatten(), 'Y'].values

#  individual effects for the untreated
effects_for_untreated = y_counterfactual_atu - untreated_df['Y']

# Calculate ATU
atu = effects_for_untreated.mean()

# --- ATE Calculation ---
# ATE is the average of all individual effects
all_effects = np.concatenate([effects_for_treated, effects_for_untreated])
ate         = all_effects.mean()

# --- MTE Calculation ---
# MTE is the max effect on the untreated
mte         = effects_for_untreated.max()

print(f"ATE: {ate}")
print(f"ATT: {att}")
print(f"ATU: {atu}")
print(f"MTE: {mte}")

ATE: 1.6952701427487222
ATT: 1.846408507179443
ATU: 1.5494765534845105
MTE: 2.1724698851117035


### Question 1: Which is closest to the average treatment effect? 

Option A
2.014

Option B
1.583

Option C
1.832

**Option D
1.695**



In [5]:
ate

1.6952701427487222

### Question 2: Which is closest to the average treatment effect on the treated? 

Option A
1.503

Option B
1.714

**Option C
1.846**

Option D
1.620

In [6]:
att

1.846408507179443

### Question 3: Which is closest to the average treatment effect on the untreated? 

Option A
1.689

Option B
2.305

Option C
1.843

**Option D
1.549**



In [7]:
atu

1.5494765534845105

### Question 4: Which is closest to the marginal treatment effect? 

Option A
1.134 

**Option B
2.172**

Option C
0.8935

Option D
1.480 



In [8]:
mte

2.1724698851117035